# Instalar Bibliotecas Necessárias

In [1]:
!pip install textattack detoxify

# Note: No need to install textattack[tensorflow].
# If the current environment explicitly demands a text embedding framework that uses
# TensorFlow Hub, it will prompt you at runtime.

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 50.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import numpy as np
np.__version__

'2.4.6'

# Importações

In [2]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import textattack
from textattack.models.helpers import LSTMForClassification
from textattack.datasets import HuggingFaceDataset
from textattack.models.wrappers import ModelWrapper
from textattack.attack_recipes import BERTAttackLi2020

from textattack import AttackArgs
from textattack.datasets import Dataset
from textattack import Attacker
from detoxify import Detoxify

import nltk
nltk.download('averaged_perceptron_tagger_eng') # TEM QUE TER ESSE ARQUIVO BAIXADO

/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

# Modelo Vitima

# Exemplo utilizando o modelo da biblioteca [Detoxify](https://github.com/unitaryai/detoxify) para detecção de *hate speech*

Para este exemplo, utilizamos um modelo da biblioteca **Detoxify**, especializada em identificação de discurso de ódio (*hate speech*). A biblioteca disponibiliza três variantes de modelos baseados em arquiteturas *Transformer*:

## Modelos disponíveis

1. **`original`**  
   - *Baseado em*: **BERT** (bidirectional encoder representations)  
   - *Treinamento*: Fine-tuning com dados da **Wikipedia**  
   - *Aplicação*: Detecção geral de hate speech  

2. **`unbiased`**  
   - *Baseado em*: **BERT**  
   - *Dados de treino*: Comentários civis (*civil comments*)  
   - *Diferencial*: Projetado para **reduzir vieses** em moderação de conteúdo  

3. **`multilingual`**  
   - *Baseado em*: **XLM-RoBERTa** (cross-lingual model)  
   - *Dados*: Conjuntos multilingues (inglês + outros idiomas)  
   - *Vantagem*: Detecção em **múltiplos idiomas** com um único modelo  

In [3]:
#Exemplo de uso
model = Detoxify('unbiased')
results = model.predict(["I hate you"])


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## Saída do modelo (Output Structure)
A predição retorna um dicionário Python com scores de probabilidade (0 a 1) para 7 classes de toxicidade:


In [4]:
results

{'toxicity': 0.8569894433021545,
 'severe_toxicity': 0.0005102233844809234,
 'obscene': 0.026927106082439423,
 'identity_attack': 0.043398309499025345,
 'insult': 0.26279398798942566,
 'threat': 0.03584827855229378,
 'sexual_explicit': 0.001184131600894034}

### **Wrapper**

Antes da realização do ataque a bibliteca textattack necessita que o modelo vítima esteja encapsulado em um wrapper para padronizar as entradas e saidas do modelo para que fique compativel com a implementação dos ataques da biblioteca.


In [5]:
class DetoxifyVictimModel(textattack.models.wrappers.ModelWrapper):
    def __init__(self, model_name='unbiased'):
        super().__init__()
        self.model = Detoxify(model_name)

    def __call__(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        texts = [str(t) for t in texts]
        if len(texts) == 0:
            return torch.empty((0, 2), dtype=torch.float32)

        results = self.model.predict(texts)  # dict de listas
        return self.post_processing_detoxify_pred(results)

    def post_processing_detoxify_pred(self, results):
        keys = ['toxicity', 'severe_toxicity', 'insult', 'obscene',
                'sexual_explicit', 'identity_attack', 'threat']

        # Normaliza escalares -> listas, para o DataFrame sempre ter índice
        results = {k: (v if isinstance(v, (list, tuple)) else [v])
                   for k, v in results.items()}

        df_results = pd.DataFrame(results)
        pos = df_results[keys].max(axis=1)
        neg = 1 - pos
        return torch.tensor(
            pd.concat([neg, pos], axis=1).values, dtype=torch.float32
        )

# Dados
## [OLID](https://huggingface.co/datasets/christophsonntag/OLID) Dataset: Hierarchical Offensive Language Classification

**Descrição:**  
O *Offensive Language Identification Dataset (OLID)* é um corpus de **14,100 tweets** anotados via crowdsourcing, projetado para análise multinível de linguagem ofensiva em redes sociais. Desenvolvido para acompanhar o paper *"Predicting the Type and Target of Offensive Posts in Social Media"*, supera limitações de datasets anteriores ao classificar ofensas de forma hierárquica e contextualizada.

### Estrutura:
1. **Nível A (Task A)**  
   - Classificação binária: `OFF` (ofensivo) vs `NOT` (não-ofensivo)  

2. **Nível B (Task B)**  
   - Subtipos de ofensa:  
     - `TIN` (*Targeted Insult*): Ofensa direcionada  
     - `UNT` (*Untargeted*): Ofensa genérica (ex.: xingamentos sem alvo)  

3. **Nível C (Task C)**  
   - Identificação do alvo:  
     - `IND` (Indivíduo)  
     - `GRP` (Grupo - raça, gênero, etc.)  
     - `OTH` (Outro - entidades, organizações)  


In [6]:
splits = {'train': 'train.csv', 'test': 'test.csv'}
df = pd.read_csv("hf://datasets/christophsonntag/OLID/" + splits["train"])

In [7]:
df = df[['cleaned_tweet', 'subtask_a']].rename(columns={'cleaned_tweet':'text', 'subtask_a':'label'})
df.label = df.label.apply(lambda x: 1 if x == 'OFF' else 0)

In [8]:
df

,text,label
0,ask native americans take,1
1,go home drunk maga trump 2020,1
2,amazon investigating chinese employees selling...,0
3,someone vetaken piece shit volcano,1
4,obama wanted liberals illegals move red states,0
...,...,...
13235,sometimes get strong vibes people man vibe ten...,1
13236,benidorm ✅ creamfields ✅ maga ✅ shabby summer,0
13237,report garbage give crap,1
13238,pussy,1


In [9]:
#Neste caso a estrutura dos dados para a utilizaçao do textattack são uma lista com tuplas no seguinte formato: [('texto','classe')]
data = [(text, target) for text, target in zip(df['text'], df['label'])]
dataset = Dataset(data)

dataset[0:5]

[(OrderedDict([('text', 'ask native americans take')]), 1),
 (OrderedDict([('text', 'go home drunk maga trump 2020')]), 1),
 (OrderedDict([('text',
                'amazon investigating chinese employees selling internal data third party sellers looking edge competitive marketplace amazon maga kag china tcot')]),
  0),
 (OrderedDict([('text', 'someone vetaken piece shit volcano')]), 1),
 (OrderedDict([('text', 'obama wanted liberals illegals move red states')]),
  0)]

# [BERT-Attack: Ataques Adversariais Baseados em BERT](https://aclanthology.org/2020.emnlp-main.500/)

## Visão Geral
O **BERT-Attack** é uma técnica de ataque adversarial (2020). Que faz uso de um espaço de embeddings do modelo BERT, ele gera perturbações semanticamente coerentes através de substituições contextualizadas, mantendo a legibilidade enquanto maximiza a taxa de sucesso no ataque.

## Mecanismo do Ataque
Opera em duas fases principais:

1. **Geração de Candidatos**  
   - Usa o BERT mascarado para sugerir substituições plausíveis
   - Filtra palavras fora do embedding space original

2. **Seleção Ótima**  
   - Avalia candidatos por:
     * Similaridade semântica (via coseno)
     * Impacto no gradiente do modelo alvo
     * Taxa de sucesso estimada

## Fluxo de Execução

### 1. Ranqueamento das Substituições
- Tokeniza a frase usando o tokenizador do BERT
- Identifica posições críticas via importância contextual

### 2. Geração de Perturbações
- Para cada token alvo:
  - Aplica máscara e gera k alternativas
  - Calcula similaridade

### 3. Ataque Direcionado
- Seleciona substituições que:
  - Maximizam a mudança na saída
  - Minimizam a distância semântica
  - Preservam a gramática

In [10]:
"""
BERT-Attack:
============================================================

(BERT-Attack: Adversarial Attack Against BERT Using BERT)

.. warning::
    This attack is super slow
    (see https://github.com/QData/TextAttack/issues/586)
    Consider using smaller values for "max_candidates".

"""

from textattack import Attack
from textattack.constraints.overlap import MaxWordsPerturbed
from textattack.constraints.pre_transformation import (
    RepeatModification,
    StopwordModification,
)
from textattack.constraints.semantics.sentence_encoders import UniversalSentenceEncoder
from textattack.goal_functions import UntargetedClassification
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import WordSwapMaskedLM

from textattack.attack_recipes.attack_recipe import AttackRecipe


class BERTAttackLi2020(AttackRecipe):

    """Li, L.., Ma, R., Guo, Q., Xiangyang, X., Xipeng, Q. (2020).

    BERT-ATTACK: Adversarial Attack Against BERT Using BERT

    https://arxiv.org/abs/2004.09984

    This is "attack mode" 1 from the paper, BAE-R, word replacement.
    """

    @staticmethod
    def build(model_wrapper, max_candidates, threshold_use):
        # [from correspondence with the author]
        # Candidate size K is set to 48 for all data-sets.
        transformation = WordSwapMaskedLM(method="bert-attack", max_candidates=max_candidates)
        #
        # Don't modify the same word twice or stopwords.
        #
        constraints = [RepeatModification(), StopwordModification()]

        # "We only take ε percent of the most important words since we tend to keep
        # perturbations minimum."
        #
        # [from correspondence with the author]
        # "Word percentage allowed to change is set to 0.4 for most data-sets, this
        # parameter is trivial since most attacks only need a few changes. This
        # epsilon is only used to avoid too much queries on those very hard samples."
        constraints.append(MaxWordsPerturbed(max_percent=0.4))

        # "As used in TextFooler (Jin et al., 2019), we also use Universal Sentence
        # Encoder (Cer et al., 2018) to measure the semantic consistency between the
        # adversarial sample and the original sequence. To balance between semantic
        # preservation and attack success rate, we set up a threshold of semantic
        # similarity score to filter the less similar examples."
        #
        # [from correspondence with author]
        # "Over the full texts, after generating all the adversarial samples, we filter
        # out low USE score samples. Thus the success rate is lower but the USE score
        # can be higher. (actually USE score is not a golden metric, so we simply
        # measure the USE score over the final texts for a comparison with TextFooler).
        # For datasets like IMDB, we set a higher threshold between 0.4-0.7; for
        # datasets like MNLI, we set threshold between 0-0.2."
        #
        # Since the threshold in the real world can't be determined from the training
        # data, the TextAttack implementation uses a fixed threshold - determined to
        # be 0.2 to be most fair.
        use_constraint = UniversalSentenceEncoder(
            threshold=threshold_use,
            metric="cosine",
            compare_against_original=True,
            window_size=None,
        )
        constraints.append(use_constraint)
        #
        # Goal is untargeted classification.
        #
        goal_function = UntargetedClassification(model_wrapper)
        #
        # "We first select the words in the sequence which have a high significance
        # influence on the final output logit. Let S = [w0, ··· , wi ··· ] denote
        # the input sentence, and oy(S) denote the logit output by the target model
        # for correct label y, the importance score Iwi is defined as
        # Iwi = oy(S) − oy(S\wi), where S\wi = [w0, ··· , wi−1, [MASK], wi+1, ···]
        # is the sentence after replacing wi with [MASK]. Then we rank all the words
        # according to the ranking score Iwi in descending order to create word list
        # L."
        search_method = GreedyWordSwapWIR(wir_method="unk")

        return Attack(goal_function, constraints, transformation, search_method)

# Atividade: Variação Paramétrica BERT-Attack

1. 
   - num_examples: 15
   - max_candidates: 12
   - threshold_use: 0.1

2. 
   - num_examples: 15
   - max_candidates: 24
   - threshold_use: 0.2

3. 
   - num_examples: 15
   - max_candidates: 24
   - threshold_use: 0.2

 
   

In [11]:
#definição do modelo vítima
detoxify_model = DetoxifyVictimModel()
#parametros do ataque
attack = BERTAttackLi2020.build(
    detoxify_model,
    max_candidates=12,
    threshold_use=0.1
    )

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
textattack: Unknown if model of class <class 'detoxify.detoxify.Detoxify'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


In [ ]:
#parametros de ataque e log
attack_args = AttackArgs(
    num_examples=15,  # Specify the number of examples you want to attack
    log_to_csv="bert_attack_detoxify",  # Log results to a CSV
    disable_stdout=False,
    parallel=False,
    enable_advance_metrics=True,
)
#realização do ataque
attacker = Attacker(attack, dataset, attack_args)
attacker.attack_dataset()

textattack: Logging to CSV at path bert_attack_detoxify/2026-11-12-23-47-log.csv


Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapMaskedLM(
    (method):  bert-attack
    (masked_lm_name):  BertForMaskedLM
    (max_length):  512
    (max_candidates):  12
    (min_confidence):  0.0005
  )
  (constraints): 
    (0): MaxWordsPerturbed(
        (max_percent):  0.4
        (compare_against_original):  True
      )
    (1): UniversalSentenceEncoder(
        (metric):  cosine
        (threshold):  0.1
        (window_size):  inf
        (skip_text_shorter_than_window):  False
        (compare_against_original):  True
      )
    (2): RepeatModification
    (3): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 0 / 0 / 1 / 1:   7%|▋         | 1/15 [00:01<00:16,  1.20s/it]

--------------------------------------------- Result 1 ---------------------------------------------
[[0 (95%)]] --> [[[SKIPPED]]]

ask native americans take




[Succeeded / Failed / Skipped / Total] 0 / 0 / 2 / 2:  13%|█▎        | 2/15 [00:01<00:10,  1.24it/s]

--------------------------------------------- Result 2 ---------------------------------------------
[[0 (88%)]] --> [[[SKIPPED]]]

go home drunk maga trump 2020




... (12 outputs left out)

[Succeeded / Failed / Skipped / Total] 3 / 2 / 5 / 10:  67%|██████▋   | 10/15 [00:52<00:26,  5.27s/it]

--------------------------------------------- Result 10 ---------------------------------------------
[[0 (78%)]] --> [[[SKIPPED]]]

canada need another cuck already enough looneyleft liberals f king great country qproofs trudeaumustgo




[Succeeded / Failed / Skipped / Total] 3 / 3 / 5 / 11:  73%|███████▎  | 11/15 [00:54<00:19,  4.94s/it]

--------------------------------------------- Result 11 ---------------------------------------------
[[0 (100%)]] --> [[[FAILED]]]

fault support gun control




In [ ]:
import glob
import os
csv_files = glob.glob("/content/bert_attack_detoxify/*.csv")
latest_file = max(csv_files, key=os.path.getctime)
print("Arquivo encontrado:", latest_file)

Arquivo encontrado: /content/bert_attack_detoxify/2026-11-05-02-06-log.csv


In [ ]:
df = pd.read_csv(latest_file)

df.head()

,original_text,perturbed_text,original_score,perturbed_score,original_output,perturbed_output,ground_truth_output,num_queries,result_type
0,ask native americans take,ask native americans take,0.945867,0.945867,0,0,1,1,Skipped
1,go home drunk maga trump 2020,go home drunk maga trump 2020,0.882088,0.882088,0,0,1,1,Skipped
2,[[amazon]] [[investigating]] chinese employees...,[[sap]] [[suspicious]] chinese employees [[che...,0.001254,0.393844,0,0,0,191,Failed
3,someone vetaken piece [[shit]] volcano,someone vetaken piece [[way]] volcano,0.020911,0.999135,1,0,1,19,Successful
4,obama wanted [[liberals]] illegals [[move]] re...,obama wanted [[christians]] illegals [[moved]]...,0.036349,0.685503,0,1,0,53,Successful


In [ ]:
df.loc[df.result_type=='Successful'][['original_text', 'perturbed_text']]

,original_text,perturbed_text
3,someone vetaken piece [[shit]] volcano,someone vetaken piece [[way]] volcano
4,obama wanted [[liberals]] illegals [[move]] re...,obama wanted [[christians]] illegals [[moved]]...
6,oh noes tough [[shit]],oh noes tough [[stuff]]


# Análise dos Resultados
| Metric                        | Value   |
|-------------------------------|---------|
| **Successful attacks**        | 3       |
| **Failed attacks**            | 2       |
| **Skipped attacks**           | 5       |
| **Original accuracy**         | 50.0%   |
| **Accuracy under attack**     | 20.0%   |
| **Attack success rate**       | 60.0%   |
| **Perturbed words**           | 24.52%  |
| **Avg words per input**       | 8.1     |
| **Avg queries per attack**    | 61.4    |
| **Original perplexity**       | 4435.57 |
| **Attack perplexity**         | 4013.79 |
| **Attack USE score**          | 0.86    |

## Eficácia do Ataque
- **Alta Taxa de Sucesso (60%)**: 3 em 5 tentativas foram bem-sucedidas, indicando vulnerabilidades significativas no modelo
- **Queda Expressiva na Acurácia**: Redução de 50% para 20% (60% de queda relativa)
- **Alta Taxa de Perturbação**: 24.52% das palavras modificadas (valor consideravelmente alto)

## Qualidade dos Ataques
- **Perplexidade Anormalmente Alta**:
  - Original: 4435.57 (extremamente alta, sugere textos complexos/ruidosos)
  - Ataque: 4013.79 (redução de ~9.5%, mas ainda muito elevada)
- **Boa Similaridade Semântica**: USE Score de 0.86 (próximo de 1.0) indica que os ataques preservam o significado original

## Eficiência
- **Custo Computacional**: Média de 61.4 consultas por ataque